# Haulio Frozen Backhaul Graph Policy

This notebook demonstrates **inference only**. The 4,393,161-parameter graph policy was trained from random initialization on synthetic snapshots and then frozen. Running these cells never updates model weights. The learned scope is one-step truck–order edge ranking; routing, anomaly handling, hard feasibility, and dispatcher acceptance remain explicit surrounding controls.

## Static inference boundary

`IoT + fleet + orders + road context → hard constraint compiler → frozen heterogeneous graph policy → conflict-free decoder + ETA gate → dispatcher review`

There is no optimizer, backward pass, auto-tuning, feedback loop, or automatic commit in the runtime bundle.

In [ ]:
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RUNTIME_PYTHON = shutil.which('python')
assert RUNTIME_PYTHON, 'Python runtime was not found'
print(f'Project root: {ROOT}')
print(f'Frozen inference runtime: {RUNTIME_PYTHON}')

In [ ]:
manifest = json.loads((ROOT / 'submission/artifacts/manifest.json').read_text(encoding='utf-8'))
artifact = ROOT / 'submission/artifacts/backhaul_policy_frozen.pt'
digest = hashlib.sha256(artifact.read_bytes()).hexdigest()
assert digest == manifest['artifact']['sha256']
{
    'model': manifest['model_name'],
    'version': manifest['model_version'],
    'sha256': digest,
    'weights_static': manifest['runtime']['weights_static'],
    'maximum_snapshot': f"{manifest['runtime']['max_trucks']} trucks × {manifest['runtime']['max_orders']} orders",
}

In [ ]:
completed = subprocess.run(
    [RUNTIME_PYTHON, 'submission/service.py', '--artifacts', 'submission/artifacts', '--input', 'submission/demo_input.json'],
    cwd=ROOT, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
result = json.loads(completed.stdout)
print(f"Inference produced {len(result['recommendations'])} recommendations and {len(result['rejected_pairs'])} reason-coded rejected pairs.")

In [ ]:
[
    {
        'truck': item['truck_id'],
        'order': item['order_id'],
        'eta_min': item['eta_minutes'],
        'score': item['policy_score'],
        'human_acceptance': item['requires_human_acceptance'],
    }
    for item in result['recommendations']
]

In [ ]:
assert result['model']['weights_static'] is True
assert result['automatic_commit'] is False
assert all(item['requires_human_acceptance'] for item in result['recommendations'])
assert len({item['truck_id'] for item in result['recommendations']}) == len(result['recommendations'])
assert len({item['order_id'] for item in result['recommendations']}) == len(result['recommendations'])
print('PASS: immutable weights, unique assignments, and mandatory human acceptance.')

In [ ]:
metrics = json.loads((ROOT / 'submission/artifacts/metrics.json').read_text(encoding='utf-8'))
{
    'evidence_boundary': 'held-out seeds from the same synthetic generator family',
    'training_seconds': round(metrics['elapsed_seconds'], 1),
    'standard_reward_gain': round(metrics['validation']['learned_vs_baseline'], 4),
    'dropout_stress_reward_gain': round(metrics['sensor_dropout_stress']['learned_vs_baseline'], 4),
    'hard_violations': int(metrics['validation']['hard_constraint_violations'] + metrics['sensor_dropout_stress']['hard_constraint_violations']),
    'duplicate_assignments': int(metrics['validation']['duplicate_assignments'] + metrics['sensor_dropout_stress']['duplicate_assignments']),
}

## Evidence limit

The reward comparison is simulator-defined. It does **not** establish real-world Haulio uplift, calibrated ETA/uncertainty, a globally optimal VRP solution, or multi-hop routing. Those claims require chronological fleet data, solver baselines, shadow deployment, and a controlled dispatcher trial.